In [3]:
year = 1993
month = 1

In [4]:
from pathlib import Path
import xarray as xr
import numpy as np
import os,subprocess, sys

In [22]:
# mesh_path = "/work/bk1450/b383184/Amazon/Mercator/data/Zgr_cmesh2.nc"
# W_path = f"/work/bk1450/b383184/Amazon/Mercator/data/variables/W_{year}-{month:02d}.nc"
# SSH_path = f"/work/bk1450/b383184/Amazon/Mercator/data/variables_c/tracers/SSH_{year}-{month:02d}c.nc"

# outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables_c/UVW/'

mesh_path = "Zgr_cmesh2.nc"
W_path = f"W_{year}-{month:02d}.nc"
SSH_path = f"SSH_{year}-{month:02d}c.nc"

outpath = "./"


In [16]:
## mesh load
ds_mesh = xr.open_dataset(mesh_path, chunks={})
ds_mesh = ds_mesh.assign_coords(x=np.arange(ds_mesh.sizes["x"]))
ds_mesh = ds_mesh.assign_coords(y=np.arange(ds_mesh.sizes["y"]))
ds_mesh = ds_mesh.assign_coords(z=np.arange(ds_mesh.sizes["z"]))
ds_mesh = ds_mesh.squeeze()

In [17]:
#Construct W depth for each water filled cell
e3t_full = xr.where(
    (ds_mesh.z + 1) <= (ds_mesh.mbathy - 1),  # wet cells above bottom cells
    ds_mesh.e3t_0,  # fill with basin wide e3t for level,
    ds_mesh.e3t_ps,  # add partial cell height otherwise
).where((ds_mesh.z + 1) <= ds_mesh.mbathy)  # remove all non-wet cells below

In [18]:
#W depths (top of cell) as vertical sum of the e3t (including the partially filled last cell above the bottom). 
#Then rename the dimension z coming from the mesh file to depthw which we'll need to align with the W file later.

depthw_ps = e3t_full.cumsum("z").where((ds_mesh.z + 1) <= ds_mesh.mbathy)
depthw_ps = depthw_ps.shift(z=1).fillna(0.0)
depthw_ps = depthw_ps.where(ds_mesh.z <= ds_mesh.mbathy)
depthw_ps = depthw_ps.rename({"z": "depthw"}).drop("depthw")

/var/folders/s1/62z527j141g8f5z22f9vb42d44rbwm/T/ipykernel_31129/2567791241.py:7: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  depthw_ps = depthw_ps.rename({"z": "depthw"}).drop("depthw")


In [19]:
## Calc height of water colum (relto $\eta=0$)
H_bottom = e3t_full.sum("z")

In [20]:
## load W file
ds_W = xr.open_dataset(W_path, chunks={"time_counter": 1})
ds_W = ds_W.assign_coords(x=np.arange(ds_W.sizes["x"]))
ds_W = ds_W.assign_coords(y=np.arange(ds_W.sizes["y"]))
ds_W = ds_W.assign_coords(z=-depthw_ps, H=H_bottom)

/Users/dlizarbe/micromamba/envs/parcels-dev/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'vovecrtz' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


In [23]:
## Load SSH (we need $\eta$)
ds_SSH = xr.open_dataset(SSH_path, engine="netcdf4",chunks={"time_counter": 1})
ds_SSH = ds_SSH.assign_coords(x=np.arange(ds_SSH.sizes["x"]))
ds_SSH = ds_SSH.assign_coords(y=np.arange(ds_SSH.sizes["y"]))

/Users/dlizarbe/micromamba/envs/parcels-dev/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'sossheig' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


## Correcting W for fixed sea surface:

$z$ is positive upward, $H$ is positive, and $\eta$ is positive upward

$$W_{merc}(z) = W_\eta(z) + W_{fixed}(z)$$

Normally,
$$W_\eta(z) = \frac{z+H}{\eta+H} \frac{\partial \eta}{\partial t}$$
but we register it to the surface and set.
$$W_\eta(z) \equiv \frac{z+H}{H} \frac{\partial \eta}{\partial t}$$
(Note the change in the denominator.)

Then
$$W_{fixed}(z) = W_{merc}(z) - W_\eta(z)$$
and
$$W_{fixed}(0) = W_{fixed}(-H) = 0$$

In [24]:
deta_dt = ds_W.vovecrtz.isel(depthw=0, drop=True)

In [25]:
eta = ds_SSH.sossheig

In [26]:
W_merc = ds_W.vovecrtz.rename("W_merc").fillna(0.0)

In [27]:
W_eta = (ds_W.z + ds_W.H) / (ds_W.H) * deta_dt
W_eta = W_eta.rename("W_eta")
W_eta = W_eta.assign_coords(z=ds_W.z)

In [28]:
W_fixed = W_merc - W_eta
W_fixed = W_fixed.rename("W_fixed")

In [29]:
## old coords
x = xr.open_dataset('U_1993-01.nc', chunks={}).x
y = xr.open_dataset('U_1993-01.nc', chunks={}).y

/Users/dlizarbe/micromamba/envs/parcels-dev/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'vozocrtx' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/Users/dlizarbe/micromamba/envs/parcels-dev/lib/python3.12/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'vozocrtx' has multiple fill values {np.float64(9.96921e+36), np.float32(9.96921e+36)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


In [30]:
#re-define the x and y to match with the U and V
W_fixed = W_fixed.assign_coords(x=x)
W_fixed = W_fixed.assign_coords(y=y)
W_fixed.name = "vovecrtz"
W_fixed = W_fixed.to_dataset()
W_fixed

<xarray.Dataset> Size: 8GB
Dimensions:       (depthw: 50, time_counter: 31, y: 499, x: 1260)
Coordinates:
  * depthw        (depthw) float32 200B 0.0 1.011 2.086 ... 5.052e+03 5.5e+03
  * time_counter  (time_counter) datetime64[ns] 248B 1993-01-01T12:00:00 ... ...
  * y             (y) int32 2kB 1375 1376 1377 1378 1379 ... 1870 1871 1872 1873
  * x             (x) int32 5kB 2306 2307 2308 2309 2310 ... 3562 3563 3564 3565
    nav_lon       (y, x) float32 3MB dask.array<chunksize=(499, 1260), meta=np.ndarray>
    nav_lat       (y, x) float32 3MB dask.array<chunksize=(499, 1260), meta=np.ndarray>
    z             (depthw, y, x) float64 251MB dask.array<chunksize=(50, 499, 1260), meta=np.ndarray>
    H             (y, x) float64 5MB dask.array<chunksize=(499, 1260), meta=np.ndarray>
Data variables:
    vovecrtz      (time_counter, depthw, y, x) float64 8GB dask.array<chunksize=(1, 25, 250, 630), meta=np.ndarray>

In [31]:
import calendar
import datetime
from datetime import date
import pandas as pd

last_day = calendar.monthrange(year, month)[1]
start_date = datetime.datetime(year, month, 1)
end_date = datetime.datetime(year, month, last_day)


## days in GLORYS
def glorys_days(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

days = glorys_days(start_date.strftime("%Y-%m-%d")
                   , end_date.strftime("%Y-%m-%d"))
#starts and ends
starts = days[0::2]
ends = days[1::2].tolist()  
ends[-1] = days[-1]

In [32]:
W_fixed = W_fixed.astype('float32')

W_out = f'W_{start_date.strftime("%Y-%m")}fc.nc'

In [33]:
W_fixed.to_netcdf(outpath+W_out)#outpath

/Users/dlizarbe/micromamba/envs/parcels-dev/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
